# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mursaleen-developer/fly-rank-ML-Intenship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My baseline rule

I will prioritize content using two observable signals:

1. **Staleness** — pages that have not been updated recently receive a higher priority.
2. **Search impressions** — pages with search visibility are more useful candidates for review.

The baseline score combines these two signals into one priority score.

Reason codes:
- `STALE_VISIBLE` — the page is stale and has meaningful search visibility.
- `STALE_LOW_VISIBILITY` — the page is stale but has low search visibility.
- `RECENT_VISIBLE` — the page has search visibility but was updated more recently.

Action labels:
- `REVIEW_REFRESH` — prioritize the page for refresh review.
- `MONITOR` — keep the page lower in the review queue.

This is a decision-support baseline. It does not claim that refreshing a page will cause better rankings or traffic.

In [10]:
# ============================================================
# ML-07 SETUP — Load data efficiently with DuckDB
# ============================================================

from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

# Load Hugging Face token
hf_token = userdata.get("HF_TOKEN")

print("HF token loaded:", hf_token is not None)

# Download March 2026 performance partition
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

# Download content metadata
content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)

print("Warehouse files ready.")

# Aggregate March data BEFORE loading into pandas
query = f"""
SELECT
    m.client_hash_id,
    m.content_hash_id,
    SUM(m.gsc_impressions) AS gsc_impressions,
    MAX(m.report_date) AS report_date,
    c.content_updated_date
FROM read_parquet('{march_path}') AS m
LEFT JOIN read_parquet('{content_path}') AS c
    ON m.client_hash_id = c.client_hash_id
    AND m.content_hash_id = c.content_hash_id
GROUP BY
    m.client_hash_id,
    m.content_hash_id,
    c.content_updated_date
"""

baseline_df = duckdb.query(query).to_df()

print("Baseline dataset shape:", baseline_df.shape)

display(baseline_df.head())

HF token loaded: True
Warehouse files ready.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline dataset shape: (331437, 5)


,client_hash_id,content_hash_id,gsc_impressions,report_date,content_updated_date
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,2026-03-31,2026-06-29
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,2026-03-31,2026-06-29
2,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,2026-03-31,2026-06-29
3,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,2026-03-31,2026-06-29
4,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,2026-03-31,2026-06-29


In [11]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

hf_token = userdata.get("HF_TOKEN")

print("HF token loaded:", hf_token is not None)

HF token loaded: True


In [12]:
from huggingface_hub import hf_hub_download

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)

print("Files downloaded successfully.")

Files downloaded successfully.


In [13]:
query = f"""
SELECT
    m.client_hash_id,
    m.content_hash_id,
    SUM(m.gsc_impressions) AS gsc_impressions,
    MAX(m.report_date) AS report_date,
    c.content_updated_date
FROM read_parquet('{march_path}') AS m
LEFT JOIN read_parquet('{content_path}') AS c
    ON m.client_hash_id = c.client_hash_id
    AND m.content_hash_id = c.content_hash_id
GROUP BY
    m.client_hash_id,
    m.content_hash_id,
    c.content_updated_date
"""

baseline_df = duckdb.query(query).to_df()

print("Baseline dataset shape:", baseline_df.shape)

display(baseline_df.head())

Baseline dataset shape: (331437, 5)


,client_hash_id,content_hash_id,gsc_impressions,report_date,content_updated_date
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,2026-03-31,2026-06-29
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,2026-03-31,2026-06-29
2,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,2026-03-31,2026-06-29
3,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,2026-03-31,2026-06-29
4,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,2026-03-31,2026-06-29


In [14]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

# 1. Load Hugging Face token
hf_token = userdata.get("HF_TOKEN")

print("HF token loaded:", hf_token is not None)

# 2. Download March 2026 performance data
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

march_df = pd.read_parquet(march_path)

print("March dataset shape:", march_df.shape)

# 3. Download content metadata
content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)

dim_content = pd.read_parquet(content_path)

print("dim_content shape:", dim_content.shape)

HF token loaded: True
March dataset shape: (9841378, 30)
dim_content shape: (519606, 26)


In [15]:
# Load content metadata

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)

dim_content = pd.read_parquet(content_path)

print("dim_content shape:", dim_content.shape)

dim_content shape: (519606, 26)


In [16]:
# Check the signals available for our baseline rule

print("GSC impressions summary:")
print(march_df["gsc_impressions"].describe())

print("\nDays since last update will come from dim_content.")

GSC impressions summary:
count    9.841378e+06
mean     2.851812e+01
std      1.559266e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      6.000000e+00
max      4.008400e+04
Name: gsc_impressions, dtype: float64

Days since last update will come from dim_content.


In [17]:
# Load content metadata

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)

dim_content = pd.read_parquet(content_path)

print("dim_content shape:", dim_content.shape)
print("\nColumns:")
print(dim_content.columns.tolist())

dim_content shape: (519606, 26)

Columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## 1. My rule and its reason codes

[TEXT CELL]
My baseline rule...
Reason codes...
Action labels...


## 2. Build the ranked queue

[CODE CELL]
Python code for building baseline score
        ↓
Top 10 output


## 3. Top-20 review

[TEXT CELL]
Your review of the top 20...


[CODE CELL]
Code/checks


## 4. Weak picks + leakage check

[TEXT CELL]
Your explanation...


[CODE CELL]
Code/checks

In [18]:
# Save the ranked baseline queue
output_path = "/content/baseline_action_score.csv"

queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "days_since_update",
        "baseline_score",
        "reason_code",
        "action"
    ]
].to_csv(output_path, index=False)

print("CSV saved:", output_path)

import numpy as np
import pandas as pd

# Work on the already-aggregated content-level data
queue = baseline_df.copy()

# Convert dates
queue["report_date"] = pd.to_datetime(queue["report_date"])
queue["content_updated_date"] = pd.to_datetime(
    queue["content_updated_date"]
)

# ------------------------------------------------------------
# Calculate staleness as it was knowable at the March decision
# moment.
#
# If an update happened after March 31, we do NOT use that
# future date. Treat the content as having no known update
# after the March decision date.
# ------------------------------------------------------------

decision_date = pd.Timestamp("2026-03-31")

queue["effective_updated_date"] = queue["content_updated_date"].where(
    queue["content_updated_date"] <= decision_date
)

# For missing/future update dates, use 0 days of known staleness
queue["days_since_update"] = (
    decision_date - queue["effective_updated_date"]
).dt.days.fillna(0)

queue["days_since_update"] = queue["days_since_update"].clip(lower=0)

# ------------------------------------------------------------
# Staleness score
# 180+ days = maximum staleness score
# ------------------------------------------------------------

queue["staleness_score"] = (
    queue["days_since_update"] / 180
).clip(0, 1)

# ------------------------------------------------------------
# Visibility score
# Log transform reduces the effect of extremely large
# impression counts.
# ------------------------------------------------------------

queue["gsc_impressions"] = queue["gsc_impressions"].fillna(0)

visibility_score = np.log1p(queue["gsc_impressions"])

max_visibility = visibility_score.max()

if max_visibility > 0:
    queue["visibility_score"] = (
        visibility_score / max_visibility
    )
else:
    queue["visibility_score"] = 0

# ------------------------------------------------------------
# Baseline priority score
#
# 60% staleness
# 40% search visibility
# ------------------------------------------------------------

queue["baseline_score"] = (
    0.60 * queue["staleness_score"] +
    0.40 * queue["visibility_score"]
)

# ------------------------------------------------------------
# Reason code
# ------------------------------------------------------------

queue["reason_code"] = np.where(
    (queue["days_since_update"] >= 180) &
    (queue["gsc_impressions"] > 0),
    "STALE_VISIBLE",
    np.where(
        queue["days_since_update"] >= 180,
        "STALE_LOW_VISIBILITY",
        "RECENT_VISIBLE"
    )
)

# ------------------------------------------------------------
# Action label
# ------------------------------------------------------------

queue["action"] = np.where(
    queue["baseline_score"] >= 0.50,
    "REVIEW_REFRESH",
    "MONITOR"
)

# ------------------------------------------------------------
# Rank the queue
# ------------------------------------------------------------

queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

print("Baseline queue shape:", queue.shape)

print("\nAction distribution:")
print(queue["action"].value_counts())

print("\nReason-code distribution:")
print(queue["reason_code"].value_counts())

print("\nTop 10:")
display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "days_since_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

CSV saved: /content/baseline_action_score.csv
Baseline queue shape: (331437, 13)

Action distribution:
action
MONITOR           326612
REVIEW_REFRESH      4825
Name: count, dtype: int64

Reason-code distribution:
reason_code
RECENT_VISIBLE          327621
STALE_LOW_VISIBILITY      3555
STALE_VISIBLE              261
Name: count, dtype: int64

Top 10:


,rank,client_hash_id,content_hash_id,gsc_impressions,days_since_update,baseline_score,reason_code,action
0,1,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,264.0,0.851772,STALE_VISIBLE,REVIEW_REFRESH
1,2,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,232.0,0.846256,STALE_VISIBLE,REVIEW_REFRESH
2,3,client_157ffe4d4a595515,content_19daa2f24df1882d,4968.0,176.0,0.842006,RECENT_VISIBLE,REVIEW_REFRESH
3,4,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,247.0,0.817971,STALE_VISIBLE,REVIEW_REFRESH
4,5,client_65de48885f4ef01b,content_eba53d72e18a9f93,734.0,231.0,0.798004,STALE_VISIBLE,REVIEW_REFRESH
5,6,client_65de48885f4ef01b,content_c126a43258b574c3,592.0,231.0,0.791563,STALE_VISIBLE,REVIEW_REFRESH
6,7,client_c182d11e4862a37d,content_5271624ae98fff86,550.0,231.0,0.789359,STALE_VISIBLE,REVIEW_REFRESH
7,8,client_65de48885f4ef01b,content_fb428c6e1ca78da4,490.0,231.0,0.785900,STALE_VISIBLE,REVIEW_REFRESH
8,9,client_65de48885f4ef01b,content_6ab29e527f71c978,486.0,234.0,0.785655,STALE_VISIBLE,REVIEW_REFRESH
9,10,client_65de48885f4ef01b,content_a9e5a8f14112ddd3,482.0,234.0,0.785408,STALE_VISIBLE,REVIEW_REFRESH


I reviewed the top 20 items from the baseline action queue.

For each item, I considered:
- Action: whether it should be reviewed for refresh or monitored.
- Reason code: why the item received its score.
- Confidence note: how strong the available signals are.
- What would make it wrong: missing data, unusual content circumstances, or a signal that does not reflect a real refresh opportunity.

The baseline is decision-support only. A high score does not prove that refreshing the content will improve performance.

In [19]:
# Display the Top 20 baseline recommendations for review

top20 = queue.head(20).copy()

display(
    top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "days_since_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

,rank,client_hash_id,content_hash_id,gsc_impressions,days_since_update,baseline_score,reason_code,action
0,1,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,264.0,0.851772,STALE_VISIBLE,REVIEW_REFRESH
1,2,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,232.0,0.846256,STALE_VISIBLE,REVIEW_REFRESH
2,3,client_157ffe4d4a595515,content_19daa2f24df1882d,4968.0,176.0,0.842006,RECENT_VISIBLE,REVIEW_REFRESH
3,4,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,247.0,0.817971,STALE_VISIBLE,REVIEW_REFRESH
4,5,client_65de48885f4ef01b,content_eba53d72e18a9f93,734.0,231.0,0.798004,STALE_VISIBLE,REVIEW_REFRESH
5,6,client_65de48885f4ef01b,content_c126a43258b574c3,592.0,231.0,0.791563,STALE_VISIBLE,REVIEW_REFRESH
6,7,client_c182d11e4862a37d,content_5271624ae98fff86,550.0,231.0,0.789359,STALE_VISIBLE,REVIEW_REFRESH
7,8,client_65de48885f4ef01b,content_fb428c6e1ca78da4,490.0,231.0,0.785900,STALE_VISIBLE,REVIEW_REFRESH
8,9,client_65de48885f4ef01b,content_6ab29e527f71c978,486.0,234.0,0.785655,STALE_VISIBLE,REVIEW_REFRESH
9,10,client_65de48885f4ef01b,content_a9e5a8f14112ddd3,482.0,234.0,0.785408,STALE_VISIBLE,REVIEW_REFRESH


### Weak picks + leakage check

Some baseline recommendations may be weak because the score is based only on simple observable signals such as visibility and content age. A high score does not guarantee that refreshing a page will improve its future performance.

Possible reasons for a weak pick include missing GSC data, unusual content behavior, or an old content update date that does not necessarily mean the content needs refreshing.

The baseline uses only information available at the decision time. It does not use future-window labels or label-derived features.

In [20]:
# Weak-pick check

print("Weak-pick review:")

weak_picks = queue[
    (queue["action"] == "REVIEW_REFRESH") &
    (queue["gsc_impressions"] <= queue["gsc_impressions"].median())
].head(10)

display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "days_since_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

# Leakage check
future_columns = [
    "future_decline_label",
    "leaked_label",
    "future_impressions",
    "future_clicks"
]

leaked_columns_present = [
    col for col in future_columns
    if col in queue.columns
]

print("\nPotential future/label-derived columns found:", leaked_columns_present)

if len(leaked_columns_present) == 0:
    print("LEAKAGE CHECK: PASS — no future/label-derived columns are present.")
else:
    print("LEAKAGE CHECK: REVIEW — remove these columns before finalizing.")


Weak-pick review:


,rank,client_hash_id,content_hash_id,gsc_impressions,days_since_update,baseline_score,reason_code,action
222,223,client_73cda7b4e4f265ea,content_01fd9554f7079c60,2.0,235.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
223,224,client_65de48885f4ef01b,content_26d4238346178145,2.0,303.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
224,225,client_65de48885f4ef01b,content_bbca8d1b7c4a1b86,2.0,303.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
225,226,client_c182d11e4862a37d,content_ca5d123a580ec67e,2.0,234.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
226,227,client_2b4306c3ed003f01,content_2a705ce668d02a9b,2.0,191.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
227,228,client_2b4306c3ed003f01,content_e8fb05a53b597afc,2.0,191.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
228,229,client_2b4306c3ed003f01,content_c986b7ab0f87ea38,2.0,191.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
229,230,client_2b4306c3ed003f01,content_c30631898f5f0396,2.0,191.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
230,231,client_65de48885f4ef01b,content_20483e735917ac4b,2.0,303.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH
231,232,client_d211cb07b9059bab,content_554e91c0f6dc1ebe,2.0,234.0,0.63296,STALE_VISIBLE,REVIEW_REFRESH



Potential future/label-derived columns found: []
LEAKAGE CHECK: PASS — no future/label-derived columns are present.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.